In [3]:
!uv pip install spacy fast_langdetect

Using Python 3.12.12 environment at: /usr
Audited 2 packages in 125ms


In [4]:
!uv run python -m spacy download en_core_web_md
!uv run python -m spacy download fr_core_news_md
!uv run python -m spacy download de_core_news_md
!uv run python -m spacy download nl_core_news_md
!uv run python -m spacy download sv_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 52.7 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 MB 41.6 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 MB 43.9 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now loa

In [5]:
import pandas as pd
df = pd.read_csv('/kaggle/input/datasets/dreamkingprince/dataset/tag1.csv')
df = df.drop(['Unnamed: 0'], axis = 1)

In [6]:
from fast_langdetect import detect

def detect_my_lang(column):
    unique = []
    for i in column:
        language = detect(i)[0]['lang']
        if language not in unique:
            unique.append(language)
    return unique

In [7]:
detect_my_lang(df['keywords'])
# In Dutch

['nl']

In [8]:
detect_my_lang(df['content'])
#fr: French nl: Dutch de: German en: This three seems main language......
# English ru: Russian sv: Swedish

['fr', 'nl', 'de', 'en', 'ru', 'sv']

In [ ]:
import re
import unicodedata

def clean_text(text):

    if not isinstance(text, str):
        return ""

    # normalize unicode
    text = unicodedata.normalize("NFKC", text)

    text = text.lower()

    text = re.sub(r"http\S+|www\S+", "", text)

    text = re.sub(r"\d+", " <num> ", text)

    text = re.sub(r"[^\w\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [10]:
df['content'] = df['content'].apply(clean_text)

In [11]:
df['content']

0      la cour constitutionnelle composc e des prc si...
1      het grondwettelijk hof samengesteld uit de voo...
2      der verfassungsgerichtshof zusammengesetzt aus...
3      la cour constitutionnelle composc e des prc si...
4      het grondwettelijk hof samengesteld uit de voo...
                             ...                        
752    het grondwettelijk hof samengesteld uit de voo...
753    het grondwettelijk hof samengesteld uit de voo...
754    der verfassungsgerichtshof zusammengesetzt aus...
755    klagen auf vc num llige oder teilweise nichtig...
756    het grondwettelijk hof samengesteld uit de voo...
Name: content, Length: 757, dtype: object

In [ ]:
import spacy

models = {
    "en": spacy.load("en_core_web_md"),
    "fr": spacy.load("fr_core_news_md"),
    "de": spacy.load("de_core_news_md"),
    "nl": spacy.load("nl_core_news_md"),
    "sv": spacy.load("sv_core_news_md")
}

In [ ]:
content_list = []

for cnt, text in enumerate(df['content']):

    try:
        lang = detect(text)[0]['lang']

        if lang not in models:
            continue

        doc = models[lang](text)

        tokens = [
            token.lemma_.lower()
            for token in doc
            if (
                not token.is_stop
                and not token.is_punct
                and not token.like_num
                and not token.is_space
                and len(token) > 2
            )
        ]

        content_list.append(tokens)

        print(cnt, "Done")

    except:
        content_list.append([])

In [ ]:
df['features'] = pd.Series(content_list)

df = df.dropna()
df['processed_text'] = df['features'].apply(lambda x: " ".join(x))

In [13]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(df['keywords'])

In [14]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from collections import Counter
import re

In [16]:
# Example
texts = df['content'].astype(str).tolist()
labels = y

In [40]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [41]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['content'],
    y,
    test_size=0.3,
    stratify=y
)



vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

train_seq = tokenizer.texts_to_sequences(X_train)
test_seq = tokenizer.texts_to_sequences(X_test)
disti = []
for i in train_seq:
    disti.append(len(i))



X_train = pad_sequences(train_seq, maxlen=max_len, padding='post')
X_test = pad_sequences(test_seq, maxlen=max_len, padding='post')
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

X_train = torch.tensor(X_train, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.long)

X_test = torch.tensor(X_test, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    num_workers=2
)
class ClassifyGRU(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_dim):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.rnn = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, x):

        x = self.embedding(x)

        _, hidden = self.rnn(x)

        hidden = hidden[-1]

        out = self.fc(hidden)

        return out

In [45]:
df = df.drop_duplicates(subset='content')

In [48]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(df['keywords'])

In [49]:
# ================================
# 1. Imports
# ================================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


# ================================
# 2. Train Test Split
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    df['content'],
    y,
    test_size=0.3,
    stratify=y,
    random_state=42
)


# ================================
# 3. Tokenization
# ================================
vocab_size = 10000
max_len = 100   # you can tune this

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

train_seq = tokenizer.texts_to_sequences(X_train)
test_seq = tokenizer.texts_to_sequences(X_test)


# ================================
# 4. Padding
# ================================
X_train = pad_sequences(train_seq, maxlen=max_len, padding='post')
X_test = pad_sequences(test_seq, maxlen=max_len, padding='post')


# ================================
# 5. Convert to Torch Tensors
# ================================
X_train = torch.tensor(X_train, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.long)

X_test = torch.tensor(X_test, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)



In [63]:

# ================================
# 6. DataLoader
# ================================
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=512,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=512,
    num_workers=2,
    pin_memory=True
)



In [64]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x: (batch, seq_len, hidden_dim)
        scores = self.attn(x)                  # (batch, seq_len, 1)
        weights = torch.softmax(scores, dim=1) # attention weights
        context = (weights * x).sum(dim=1)     # weighted sum
        return context


class AdvancedGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()

        # 🔤 Embedding
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # 🔁 Multi-layer Bidirectional GRU
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=0.5,
            bidirectional=True
        )

        # 📏 Because bidirectional → hidden_dim * 2
        self.layer_norm = nn.LayerNorm(hidden_dim * 2)

        # 🔥 Attention
        self.attention = Attention(hidden_dim * 2)

        # 🧠 Deep Classifier
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.5),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(128, 2)
        )

    def forward(self, x):
        # Embedding
        x = self.embedding(x)  # (batch, seq_len, embed_dim)

        # GRU output
        out, _ = self.gru(x)   # (batch, seq_len, hidden*2)

        # Normalize
        out = self.layer_norm(out)

        # Attention pooling
        context = self.attention(out)  # (batch, hidden*2)

        # Classification
        out = self.fc(context)

        return out

In [67]:

# ================================
# 7. Device (GPU)
# ================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.backends.cudnn.benchmark = True




# ================================
# 9. Initialize Model
# ================================
model = AdvancedGRU(vocab_size, embed_dim=128, hidden_dim=256)
model = model.to(device)


# ================================
# 10. Loss & Optimizer
# ================================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


# ================================
# 11. Mixed Precision (Optional Boost)
# ================================
scaler = torch.cuda.amp.GradScaler()




Using device: cuda


/tmp/ipykernel_247/2078897687.py:29: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [68]:
EPOCHS = 100

for epoch in range(EPOCHS):
    
    # ======================
    # TRAINING
    # ======================
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

        # 🔥 Calculate accuracy
        _, predicted = torch.max(outputs, 1)
        train_total += y_batch.size(0)
        train_correct += (predicted == y_batch).sum().item()


    train_acc = train_correct / train_total
    train_loss = train_loss / len(train_loader)


    # ======================
    # EVALUATION
    # ======================
    model.eval()
    test_loss = 0
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            test_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            test_total += y_batch.size(0)
            test_correct += (predicted == y_batch).sum().item()

    test_acc = test_correct / test_total
    test_loss = test_loss / len(test_loader)


    # ======================
    # PRINT RESULTS
    # ======================
    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Test  Loss: {test_loss:.4f} | Test  Acc: {test_acc:.4f}")
    print("-" * 50)

/tmp/ipykernel_247/834914667.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1
Train Loss: 0.7416 | Train Acc: 0.5176
Test  Loss: 0.6615 | Test  Acc: 0.6393
--------------------------------------------------
Epoch 2
Train Loss: 0.6343 | Train Acc: 0.6197
Test  Loss: 0.6529 | Test  Acc: 0.6393
--------------------------------------------------
Epoch 3
Train Loss: 0.6090 | Train Acc: 0.6761
Test  Loss: 0.6330 | Test  Acc: 0.6557
--------------------------------------------------
Epoch 4
Train Loss: 0.5427 | Train Acc: 0.7148
Test  Loss: 0.6070 | Test  Acc: 0.6639
--------------------------------------------------
Epoch 5
Train Loss: 0.4946 | Train Acc: 0.8063
Test  Loss: 0.5728 | Test  Acc: 0.6967
--------------------------------------------------
Epoch 6
Train Loss: 0.4250 | Train Acc: 0.8345
Test  Loss: 0.5370 | Test  Acc: 0.7213
--------------------------------------------------
Epoch 7
Train Loss: 0.3562 | Train Acc: 0.8662
Test  Loss: 0.4971 | Test  Acc: 0.7131
--------------------------------------------------
Epoch 8
Train Loss: 0.2944 | Train Acc: 0